第二节，Langchain Express Language

In [1]:
import os
import openai

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
openai.api_key = os.getenv("OPENAI_API_KEY")

In [2]:
from langchain_classic.prompts import ChatPromptTemplate
from langchain_classic.chat_models import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

In [3]:
prompts = ChatPromptTemplate.from_template(
    "给我讲个关于 {topic} 的笑话",
)

In [4]:
model = ChatOpenAI(
    model="LongCat-Flash-Chat",
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.longcat.chat/openai"
    )

parser = StrOutputParser()
chain = prompts | model | parser
print(chain.invoke({"topic": "Langchain"}))

C:\Users\35186\AppData\Local\Temp\ipykernel_26724\184424102.py:1: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the `langchain-openai package and should be used instead. To use it run `pip install -U `langchain-openai` and import as `from `langchain_openai import ChatOpenAI``.
  model = ChatOpenAI(


当然！来一个程序员专属的 LangChain 笑话：

---

**为什么 LangChain 从不迷路？**

因为它总是能 **chain** 住上下文！🚀

（而且它的文档说：“别担心，我们帮你把 prompt 串起来，不会断链！”）

---

再来一个冷一点的：

> 一个 LLM 对 LangChain 说：“我感觉自己像个提词器。”  
> LangChain 回答：“不，你是我的 **agent** —— 但别想拿工资。”

😄

（P.S. 真实情况：LangChain 确实让 LLM 变得更“智能”了，至少不会在对话中突然失忆……希望如此！）


用lcel复现流程

In [5]:
from langchain_classic.embeddings import OpenAIEmbeddings
from langchain_classic.vectorstores import DocArrayInMemorySearch

In [6]:
from langchain_community.embeddings import DashScopeEmbeddings

In [7]:
embeddings = DashScopeEmbeddings(
    model="text-embedding-v1",  # 通义文本向量常用模型名
    dashscope_api_key=None,     # 不填则默认走环境变量 DASHSCOPE_API_KEY
)

In [8]:
vectorstore = DocArrayInMemorySearch.from_texts(
    texts=[
        "Langchain 是一个用于构建语言模型的框架", 
        "Veinsure 是一个大学生", 
        "Veinsure 在赛博理塘读书"],
    embedding=embeddings,
)


In [9]:
retriever = vectorstore.as_retriever()

In [10]:
retriever.invoke("Veinsure 是谁")

[Document(metadata={}, page_content='Veinsure 是一个大学生'),
 Document(metadata={}, page_content='Veinsure 在赛博理塘读书'),
 Document(metadata={}, page_content='Langchain 是一个用于构建语言模型的框架')]

In [11]:
retriever.invoke("Langchain是什么")

[Document(metadata={}, page_content='Langchain 是一个用于构建语言模型的框架'),
 Document(metadata={}, page_content='Veinsure 是一个大学生'),
 Document(metadata={}, page_content='Veinsure 在赛博理塘读书')]

In [12]:
retriever.invoke("赛博理塘有什么人")

[Document(metadata={}, page_content='Veinsure 在赛博理塘读书'),
 Document(metadata={}, page_content='Langchain 是一个用于构建语言模型的框架'),
 Document(metadata={}, page_content='Veinsure 是一个大学生')]

In [13]:
template = """根据以下内容回答问题：
{context}

问题：{question}
"""

prompt = ChatPromptTemplate.from_template(template)

In [14]:
from langchain_core.runnables import RunnableMap

In [15]:
chain = RunnableMap({
    "context": lambda x: retriever.invoke(x["question"]),
    "question": lambda x: x["question"],
}) | prompt | model | parser


In [16]:
chain.invoke({"question": "Veinsure 是谁"})

'Veinsure 是一名大学生，目前在赛博理塘读书。'

In [17]:
inputs = RunnableMap({
    "context": lambda x: retriever.invoke(x["question"]),
    "question": lambda x: x["question"],
})

In [18]:
inputs.invoke({"question": "Veinsure 是谁"})

{'context': [Document(metadata={}, page_content='Veinsure 是一个大学生'),
  Document(metadata={}, page_content='Veinsure 在赛博理塘读书'),
  Document(metadata={}, page_content='Langchain 是一个用于构建语言模型的框架')],
 'question': 'Veinsure 是谁'}

In [19]:
functions = [
    {
        "name": "get_current_weather",
        "description": "Get the current weather given a airport code",
        "parameters": {
            "type": "object",
            "properties": {
                "airport_code": {
                    "type": "string",
                    "description": "The airport code to get the weather for",
                },
            },
            "required": ["airport_code"],
        },
    }
]

In [20]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
    ]
)

In [21]:
model = ChatOpenAI(
    model="LongCat-Flash-Chat",
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.longcat.chat/openai"
).bind(functions=functions)

In [22]:
runnable = prompt | model
result = runnable.invoke({"input": "What is the weather in San Francisco?"})
print(result)

content="I can't provide real-time weather updates. To check the current weather in San Francisco, I recommend using a reliable weather service like:\n\n- [Weather.com (The Weather Channel)](https://weather.com)\n- [National Weather Service – San Francisco](https://www.weather.gov/mtr/)\n- [AccuWeather – San Francisco](https://www.accuweather.com)\n- Your smartphone’s weather app\n\nSan Francisco is known for its microclimates—cool, foggy mornings and evenings are common, especially in summer—so conditions can vary significantly across different neighborhoods!" additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 121, 'prompt_tokens': 19, 'total_tokens': 140, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'cache_write_tokens': 0, 'cache_read_tokens': 0, 'input_tokens': 0, 'output_tokens': 0, 'cached_tokens': 0}, 'model_name': 'LongCat-Flash-Chat', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019cffe3-4617-